# Setup data

In [14]:
import torch
from scipy.special import softmax

def load_params(model_path, weights_only=False):
    # 1) Load parameters
    model_data = torch.load(model_path, map_location=torch.device('cpu'), weights_only=weights_only)
    theta = model_data['theta']          # shape: (n_people, k)
    d = model_data['d']                  # shape: (n_items,  k)
    p = model_data['phi']

    print("--- Initial Loaded Data ---")
    print(f"Original theta shape: {theta.shape}")
    print(f"Original 'd' matrix shape: {d.shape}\n")

    d_numpy = d.detach().cpu().numpy()
    theta_numpy = theta.detach().cpu().numpy()
    p_numpy = p.detach().cpu().numpy()
    w_numpy = softmax(p_numpy, axis=1)
    return d_numpy, theta_numpy, p_numpy, w_numpy

In [15]:
import pandas as pd
import numpy as np
import sys
from pathlib import Path
sys.path.append('..')
import style
# Concatenate math and gsm outputs for theta, a, b

d, theta, p, w = load_params('../result/lada-fitting-joint/lada_joint_k2_legalbench.pt', weights_only=False)

resmat = pd.read_pickle('../data-reeval-multi/resmat.pkl')

conv = ['legalbench']
conv_mask = resmat.loc[:, resmat.columns.get_level_values("scenario").isin(conv)]

conv_questions = conv_mask.columns.get_level_values('input.text').tolist()

answers_path = Path('../data-processing/legalbench/legalbench_result.pkl')
if not answers_path.exists():
    raise FileNotFoundError(f'Expected answer key at {answers_path} was not found. Run the assembly script first.')
answer_df = pd.read_pickle(answers_path)
answer_frame = answer_df.loc['answer'].rename('answer').reset_index()
type_frame = answer_df.loc['type'].rename('type').reset_index()
answer_frame['scenario'] = answer_frame['scenario'].astype(str).str.lower()
type_frame['scenario'] = type_frame['scenario'].astype(str).str.lower()
merged_answers = answer_frame.merge(
    type_frame[['scenario', 'input.text', 'type']],
    on=['scenario', 'input.text'],
    how='left'
 )
answer_map = merged_answers[merged_answers['scenario'] == 'legalbench'].set_index('input.text')[['answer', 'type']]
answers_aligned = answer_map.reindex(conv_questions)
if answers_aligned['answer'].isna().any() or answers_aligned['type'].isna().any():
    missing_answers = answers_aligned[answers_aligned['answer'].isna()].index.tolist()
    missing_types = answers_aligned[answers_aligned['type'].isna()].index.tolist()
    raise ValueError(
        'Missing answers or types for prompts. '
        f'Missing answers: {missing_answers[:5]} | Missing types: {missing_types[:5]}'
    )

theta_df = pd.DataFrame(theta, columns=[f'Factor_{i+1}' for i in range(theta.shape[1])], index=resmat.index)
w_df = pd.DataFrame(w, columns=[f'Factor_{i+1}' for i in range(w.shape[1])])
w_df['question'] = conv_questions
w_df['answer'] = answers_aligned['answer'].values
w_df['type'] = answers_aligned['type'].values

models_answered = conv_mask.dropna(how='all').index.tolist()
accuracy = (conv_mask.sum(axis=1) / conv_mask.notnull().sum(axis=1)).loc[models_answered]
theta_irt = pd.read_csv('../result/irt-fitting/calibration_result_theta_legalbench.csv')
theta_df = theta_df.loc[models_answered]
theta_df['accuracy'] = accuracy
theta_df['irt'] = theta_irt.values

--- Initial Loaded Data ---
Original theta shape: torch.Size([183, 2])
Original 'd' matrix shape: torch.Size([1997])



In [16]:
import os

os.makedirs('../output/lada', exist_ok=True)

export_cols = ['Factor_1', 'Factor_2', 'question', 'answer', 'type']
available_cols = [col for col in export_cols if col in w_df.columns]
w_export = w_df[available_cols]
w_export.to_csv(f'../output/lada/legalbench_lada_2k.csv', index=False)
w_export.sort_values(by=['Factor_1'], ascending=False).to_csv(f'../output/lada/legalbench_lada_2k_f1_desc.csv', index=False)
w_export.sort_values(by=['Factor_2'], ascending=False).to_csv(f'../output/lada/legalbench_lada_2k_f2_desc.csv', index=False)

# Analysis

In [17]:
import pandas as pd

df = pd.read_csv(f'../output/lada/legalbench_lada_2k.csv')

In [18]:
df.sort_values(ascending=False, by='Factor_1')

,Factor_1,Factor_2,question,answer,type
812,0.994935,0.005065,Question: Consider the country of Dominica. Do...,No,international_citizenship
1038,0.994394,0.005606,Question: Consider the country of Kyrgyzstan. ...,No,international_citizenship
1081,0.994392,0.005608,Question: Consider the country of Liechtenstei...,No,international_citizenship
1313,0.994289,0.005711,Question: Consider the country of Saint Lucia....,No,international_citizenship
876,0.994158,0.005842,Question: Consider the country of Gabon. Does ...,No,international_citizenship
...,...,...,...,...,...
960,0.007328,0.992672,Question: Consider the country of Iran. Does t...,Yes,international_citizenship
1008,0.007136,0.992864,Question: Consider the country of Kazakhstan. ...,Yes,international_citizenship
1412,0.006846,0.993154,Question: Consider the country of Sudan. Does ...,Yes,international_citizenship
637,0.006218,0.993781,Question: Consider the country of Azerbaijan. ...,Yes,international_citizenship
